In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ── Load and construct ceiling ─────────────────────────────────────────────
df = pd.read_csv('../data/gold_policy_clean.csv', parse_dates=['date'])
df['ceiling'] = df['parity_post'] - df['parity_pre']

cols = ['date','domestic_premium','post_hike','t',
        'delta_Gold_USD','delta_FX','ceiling','pre_restriction']
its = df[cols].copy()

# ── PRIMARY sample (matches Notebook 03 main spec) ─────────────────────────
its_primary = its[(its['date'] >= '2024-07-24') | (its['post_hike'] == 1)].dropna().copy()

# ── FULL window (for window sensitivity cell) ──────────────────────────────
its_full = its.dropna().copy()

# ── Primary spec parameters ────────────────────────────────────────────────
T          = len(its_primary)
NW_LAG     = int(np.ceil(0.75 * T**(1/3)))
# Load ITS headline result from shared cache (written by 03_causal.ipynb)
import json as _json
with open('../data/its_results.json') as _f:
    _cache = _json.load(_f)
BETA1_MAIN   = _cache['ITS_BETA1']     # headline β₁ from Notebook 03
mean_ceiling = _cache['mean_ceiling']  # post-hike ceiling (for PT calc)

y      = its_primary['domestic_premium']
X_main = sm.add_constant(its_primary[['post_hike','t','delta_Gold_USD','delta_FX']])

print(f'Primary sample: N={T}, NW lag={NW_LAG}')
print(f'Full window:    N={len(its_full)}')
print(f'Main result to stress-test: β₁=₹{BETA1_MAIN:,.0f}')

Primary sample: N=375, NW lag=6
Full window:    N=852
Main result to stress-test: β₁=₹9,841


In [2]:
# Cell 2 — Placebo test
fake_dates = {
    'Nov 1 2025':         '2025-11-01',
    'Jan 15 2026':        '2026-01-15',
    'Mar 1 2026':         '2026-03-01',
    'May 13 2026 (REAL)': '2026-05-13',
}

placebo_results = []

for label, cutoff in fake_dates.items():
    its_primary['post_fake'] = (its_primary['date'] >= cutoff).astype(int)
    X_fake = sm.add_constant(
        its_primary[['post_fake','t','delta_Gold_USD','delta_FX']]
    )
    res = sm.OLS(y, X_fake).fit(cov_type='HAC', cov_kwds={'maxlags': NW_LAG})
    placebo_results.append({
        'Date':         label,
        'β₁':          res.params['post_fake'],
        'NW SE':        res.bse['post_fake'],
        'p-value':      res.pvalues['post_fake'],
        'Significant?': 'YES ***' if res.pvalues['post_fake'] < 0.001 else
                        'YES *'   if res.pvalues['post_fake'] < 0.05  else 'NO'
    })

plac = pd.DataFrame(placebo_results)
print('PLACEBO TEST RESULTS')
print('=' * 70)
print(plac.to_string(index=False, float_format=lambda x: f'{x:,.1f}'))

PLACEBO TEST RESULTS
              Date      β₁   NW SE  p-value Significant?
        Nov 1 2025  -931.6 1,071.9      0.4           NO
       Jan 15 2026 2,004.1 1,252.7      0.1           NO
        Mar 1 2026 2,908.6 1,557.2      0.1           NO
May 13 2026 (REAL) 9,840.5   516.1      0.0      YES ***


In [3]:
# Cell 3 — NW lag sensitivity
lags = [3, 6, 8, 10, 20]
lag_results = []

for lag in lags:
    res = sm.OLS(y, X_main).fit(cov_type='HAC', cov_kwds={'maxlags': lag})
    ci  = res.conf_int().loc['post_hike']
    lag_results.append({
        'NW lag':   lag,
        'β₁':      res.params['post_hike'],
        'NW SE':    res.bse['post_hike'],
        'CI lower': ci[0],
        'CI upper': ci[1],
        'p-value':  res.pvalues['post_hike'],
        'Note':     '← main spec' if lag == NW_LAG else ''
    })

lag_df = pd.DataFrame(lag_results)
print('NW LAG SENSITIVITY')
print('=' * 80)
print(lag_df.to_string(index=False, float_format=lambda x: f'{x:,.1f}'))

NW LAG SENSITIVITY
 NW lag      β₁  NW SE  CI lower  CI upper  p-value        Note
      3 9,840.5  454.5   8,949.6  10,731.4      0.0            
      6 9,840.5  516.1   8,828.9  10,852.2      0.0 ← main spec
      8 9,840.5  537.1   8,787.8  10,893.3      0.0            
     10 9,840.5  550.5   8,761.6  10,919.4      0.0            
     20 9,840.5  546.2   8,770.0  10,911.0      0.0            


In [4]:
# Cell 4 — Window sensitivity
windows = {
    'Jan 2022 (full)':        '2022-01-01',
    'Jan 2024':               '2024-01-01',
    'Jul 2024 (primary)':     '2024-07-24',
}

win_results = []

for label, start in windows.items():
    subset  = its_full[its_full['date'] >= start].copy()
    nw_w    = int(np.ceil(0.75 * len(subset)**(1/3)))
    y_w     = subset['domestic_premium']
    X_w     = sm.add_constant(subset[['post_hike','t','delta_Gold_USD','delta_FX']])
    res     = sm.OLS(y_w, X_w).fit(cov_type='HAC', cov_kwds={'maxlags': nw_w})
    ci      = res.conf_int().loc['post_hike']
    win_results.append({
        'Window':   label,
        'N':        len(subset),
        'NW lag':   nw_w,
        'β₁':      res.params['post_hike'],
        'NW SE':    res.bse['post_hike'],
        'CI lower': ci[0],
        'CI upper': ci[1],
    })

win_df = pd.DataFrame(win_results)
print('WINDOW SENSITIVITY')
print('=' * 85)
print(win_df.to_string(index=False, float_format=lambda x: f'{x:,.1f}'))

WINDOW SENSITIVITY
            Window   N  NW lag       β₁  NW SE  CI lower  CI upper
   Jan 2022 (full) 852       8 10,257.4  520.7   9,236.9  11,277.9
          Jan 2024 488       6 11,771.3  585.9  10,622.9  12,919.7
Jul 2024 (primary) 375       6  9,840.5  516.1   8,828.9  10,852.2


In [5]:
# Cell 5 — Anticipation test
# Drop 5 trading days immediately before May 13
pre13  = its_primary[its_primary['date'] < '2026-05-13']['date'].sort_values()
drop_5 = pre13.iloc[-5:]

no_ant = its_primary[~its_primary['date'].isin(drop_5)].copy()
y_a    = no_ant['domestic_premium']
X_a    = sm.add_constant(no_ant[['post_hike','t','delta_Gold_USD','delta_FX']])
res_a  = sm.OLS(y_a, X_a).fit(cov_type='HAC', cov_kwds={'maxlags': NW_LAG})

print('ANTICIPATION TEST')
print('=' * 50)
print(f'Dropped dates:  {drop_5.dt.date.tolist()}')
print(f'N after drop:   {len(no_ant)} (was {T})')
print()
print(f'β₁ (no anticipation window):  ₹{res_a.params["post_hike"]:,.1f}')
print(f'β₁ (main spec):               ₹{BETA1_MAIN:,.1f}')
print(f'Difference:                   ₹{res_a.params["post_hike"] - BETA1_MAIN:,.1f}')
print(f'p-value:                      {res_a.pvalues["post_hike"]:.2e}')

ANTICIPATION TEST
Dropped dates:  [datetime.date(2026, 5, 6), datetime.date(2026, 5, 7), datetime.date(2026, 5, 8), datetime.date(2026, 5, 11), datetime.date(2026, 5, 12)]
N after drop:   370 (was 375)

β₁ (no anticipation window):  ₹9,790.0
β₁ (main spec):               ₹9,840.5
Difference:                   ₹-50.5
p-value:                      1.33e-71


In [6]:
# Cell 6 — pre_restriction control
X_restr = sm.add_constant(
    its_primary[['post_hike','t','delta_Gold_USD','delta_FX','pre_restriction']]
)
res_r = sm.OLS(y, X_restr).fit(cov_type='HAC', cov_kwds={'maxlags': NW_LAG})

print('PRE_RESTRICTION CONTROL')
print('=' * 55)
print(f'β₁ (main spec):              ₹{BETA1_MAIN:,.1f}')
print(f'β₁ (+ pre_restriction):      ₹{res_r.params["post_hike"]:,.1f}')
print(f'Difference:                  ₹{res_r.params["post_hike"] - BETA1_MAIN:,.1f}')
print(f'β pre_restriction:           ₹{res_r.params["pre_restriction"]:,.1f}')
print(f'p (pre_restriction):         {res_r.pvalues["pre_restriction"]:.3f}')
print(f'p (post_hike):               {res_r.pvalues["post_hike"]:.2e}')

PRE_RESTRICTION CONTROL
β₁ (main spec):              ₹9,840.5
β₁ (+ pre_restriction):      ₹9,757.9
Difference:                  ₹-82.7
β pre_restriction:           ₹-279.6
p (pre_restriction):         0.617
p (post_hike):               2.52e-53


In [7]:
# Cell — BullionWorld source robustness
df_raw = pd.read_csv('../data/gold_policy_clean.csv', parse_dates=['date'])

# --- BullionWorld augmented sample (N=368) ---
df_bw = df_raw[
    (df_raw['date'] >= '2024-07-24')
].dropna(subset=['domestic_premium','post_hike','t','delta_Gold_USD','delta_FX']).copy()
T_bw  = len(df_bw)
NW_bw = int(np.ceil(0.75 * T_bw**(1/3)))
y_bw  = df_bw['domestic_premium']
X_bw  = sm.add_constant(df_bw[['post_hike','t','delta_Gold_USD','delta_FX']])
m_bw  = sm.OLS(y_bw, X_bw).fit(cov_type='HAC', cov_kwds={'maxlags': NW_bw})
b1_bw = m_bw.params['post_hike']

# --- IBJA-PDF only sample (N=338, primary) ---
prim_ibja = df_raw[
    (df_raw['date'] >= '2024-07-24') &
    (df_raw['ibja_source'] != 'bullionworld')
].dropna(subset=['domestic_premium','post_hike','t','delta_Gold_USD','delta_FX']).copy()
T_ibja  = len(prim_ibja)
NW_ibja = int(np.ceil(0.75 * T_ibja**(1/3)))
y_ibja  = prim_ibja['domestic_premium']
X_ibja  = sm.add_constant(prim_ibja[['post_hike','t','delta_Gold_USD','delta_FX']])
m_ibja  = sm.OLS(y_ibja, X_ibja).fit(cov_type='HAC', cov_kwds={'maxlags': NW_ibja})
b1_ibja = m_ibja.params['post_hike']

# Use PRIMARY ceiling (11,989) for both PT figures — apples-to-apples comparison
primary_ceiling = mean_ceiling  # computed in cell 0 from 31 post-hike obs
pt_bw   = b1_bw   / primary_ceiling * 100
pt_ibja = b1_ibja / primary_ceiling * 100
pct_diff = (b1_ibja - b1_bw) / b1_bw * 100

print('=== BullionWorld Source Robustness ===')
print(f'Primary (all sources):  N={T_bw}  β₁={b1_bw:,.0f}  PT={pt_bw:.1f}%  (vs primary ceiling Rs.{primary_ceiling:,.0f})')
print(f'IBJA-PDF only:          N={T_ibja}  β₁={b1_ibja:,.1f}  SE={m_ibja.bse["post_hike"]:,.1f}  PT={pt_ibja:.1f}%')
print(f'Change in β₁: {pct_diff:+.1f}%  (NW lag={NW_ibja})')
print()
print(f'Conclusion: BullionWorld augmented sample gives {pt_bw:.1f}% PT vs {pt_ibja:.1f}% IBJA-only — '
      f'difference of {pt_ibja-pt_bw:.1f}pp, within primary CI. Source splice is benign.')


=== BullionWorld Source Robustness ===
Primary (all sources):  N=375  β₁=9,841  PT=81.7%  (vs primary ceiling Rs.12,039)
IBJA-PDF only:          N=336  β₁=10,124.4  SE=459.0  PT=84.1%
Change in β₁: +2.9%  (NW lag=6)

Conclusion: BullionWorld augmented sample gives 81.7% PT vs 84.1% IBJA-only — difference of 2.4pp, within primary CI. Source splice is benign.


In [8]:
# Cell 7 — Robustness summary table
summary = {
    'Test': [
        f'Main spec (NW lag={NW_LAG}, Jul 2024+)',
        'Placebo — Nov 1 2025',
        'Placebo — Jan 15 2026',
        'Placebo — Mar 1 2026',
        'NW lag = 3',
        'NW lag = 8',
        'NW lag = 10',
        'NW lag = 20',
        'Window: Jan 2022 (full)',
        'Window: Jan 2024',
        'Window: Jul 2024 (primary)',
        'Anticipation test (drop 5 days)',
        '+ pre_restriction control',
    ],
    'β₁ (₹)': [
        BETA1_MAIN,
        plac.loc[plac['Date']=='Nov 1 2025',         'β₁'].values[0],
        plac.loc[plac['Date']=='Jan 15 2026',        'β₁'].values[0],
        plac.loc[plac['Date']=='Mar 1 2026',         'β₁'].values[0],
        lag_df.loc[lag_df['NW lag']==3,  'β₁'].values[0],
        lag_df.loc[lag_df['NW lag']==8,  'β₁'].values[0],
        lag_df.loc[lag_df['NW lag']==10, 'β₁'].values[0],
        lag_df.loc[lag_df['NW lag']==20, 'β₁'].values[0],
        win_df.loc[win_df['Window']=='Jan 2022 (full)',    'β₁'].values[0],
        win_df.loc[win_df['Window']=='Jan 2024',           'β₁'].values[0],
        win_df.loc[win_df['Window']=='Jul 2024 (primary)', 'β₁'].values[0],
        res_a.params['post_hike'],
        res_r.params['post_hike'],
    ],
    'Significant?': [
        'YES ***', 'NO', 'NO', 'NO',
        'YES ***', 'YES ***', 'YES ***', 'YES ***',
        'YES ***', 'YES ***', 'YES ***',
        'YES ***', 'YES ***',
    ]
}

s = pd.DataFrame(summary)
print('ROBUSTNESS SUMMARY — Notebook 05')
print(f'Dependent variable: domestic_premium (₹/10g)')
print(f'Main result: β₁=₹{BETA1_MAIN:,.0f}, pass-through=79.4%')
print('=' * 60)
print(s.to_string(index=False, float_format=lambda x: f'{x:,.0f}'))

ROBUSTNESS SUMMARY — Notebook 05
Dependent variable: domestic_premium (₹/10g)
Main result: β₁=₹9,841, pass-through=79.4%
                           Test  β₁ (₹) Significant?
Main spec (NW lag=6, Jul 2024+)   9,841      YES ***
           Placebo — Nov 1 2025    -932           NO
          Placebo — Jan 15 2026   2,004           NO
           Placebo — Mar 1 2026   2,909           NO
                     NW lag = 3   9,841      YES ***
                     NW lag = 8   9,841      YES ***
                    NW lag = 10   9,841      YES ***
                    NW lag = 20   9,841      YES ***
        Window: Jan 2022 (full)  10,257      YES ***
               Window: Jan 2024  11,771      YES ***
     Window: Jul 2024 (primary)   9,841      YES ***
Anticipation test (drop 5 days)   9,790      YES ***
      + pre_restriction control   9,758      YES ***
